In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import os

# ================= Global Configuration =================
# Must strictly match the dimensions used during dataset construction
TARGET_SIZE = (1280, 720) 
# ========================================================

def extract_features_and_visualize(image_path):
    """
    Extract features and return image data for visualization.
    Includes illumination compensation algorithm.
    """
    img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), -1)
    if img is None:
        return None, None, None, None, None
    
    # Resize image to target dimensions
    img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    # --- Illumination Compensation ---
    # 1. Estimate background illumination distribution
    bg_illumination = cv2.GaussianBlur(gray, (151, 151), 0)
    
    # 2. Background subtraction
    diff = cv2.subtract(bg_illumination, gray)
    
    # 3. Contrast enhancement
    diff = cv2.convertScaleAbs(diff, alpha=1.5, beta=0)
    
    # 4. Fixed threshold binarization
    _, binary_mask = cv2.threshold(diff, 20, 255, cv2.THRESH_BINARY)
    
    # 5. Morphological opening to remove isolated noise
    kernel = np.ones((3, 3), np.uint8)
    clean_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)
    # ---------------------------------
    
    # Calculate basic features
    debris_area = np.sum(clean_mask == 255)
    total_pixels = TARGET_SIZE[0] * TARGET_SIZE[1]
    area_ratio = debris_area / total_pixels
    
    inverted_gray = 255 - gray
    debris_density = np.sum(inverted_gray[clean_mask == 255])
    
    # Connected components analysis
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(clean_mask, connectivity=8)
    max_cluster_area = 0
    core_agglomeration_ratio = 0.0
    max_cluster_bbox = None 
    
    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        max_idx = np.argmax(areas) + 1 
        
        max_cluster_area = stats[max_idx, cv2.CC_STAT_AREA]
        max_cluster_bbox = stats[max_idx, :4] 
        
        large_clusters_area = np.sum(areas[areas > 50])
        if debris_area > 0:
            core_agglomeration_ratio = large_clusters_area / debris_area
            
    features = [area_ratio, debris_density, max_cluster_area, core_agglomeration_ratio]
    return features, img_resized, clean_mask, max_cluster_bbox

def train_and_save_model(csv_path, model_save_path, scaler_save_path):
    """
    Load dataset, train the Random Forest model, and save weights.
    """
    print("Loading dataset and training model...")
    df = pd.read_csv(csv_path)
    
    feature_cols = ['总面积占比(Area_Ratio)', '累积灰度密度(IOD)', '最大聚集块面积(Max_Cluster)', '大块占比(Core_Ratio)']
    X = df[feature_cols]
    y = df['Wear_Level']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))
    print(f"Model training completed. Validation accuracy: {acc * 100:.2f}%\n")
    
    Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)
    
    joblib.dump(rf_model, model_save_path)
    joblib.dump(scaler, scaler_save_path)

def test_and_visualize_unseen(unseen_dir, output_vis_dir, model_path, scaler_path):
    """
    Predict on unseen images and generate visualization results.
    """
    print("Testing and generating visualizations...")
    rf_model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    
    unseen_path = Path(unseen_dir)
    vis_path = Path(output_vis_dir)
    vis_path.mkdir(parents=True, exist_ok=True)
    
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    
    for file_path in unseen_path.iterdir():
        if file_path.suffix.lower() not in valid_extensions:
            continue
            
        # 1. Extract features and image data
        features, img_resized, clean_mask, max_bbox = extract_features_and_visualize(str(file_path))
        if features is None:
            continue
            
        # 2. Model prediction
        features_scaled = scaler.transform([features])
        prediction = rf_model.predict(features_scaled)[0]
        
        # 3. Visualization
        vis_img = img_resized.copy()
        
        # Draw green contours for all debris
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis_img, contours, -1, (0, 255, 0), 1)
        
        # Highlight the maximum cluster with a red bounding box
        if max_bbox is not None:
            x, y, w, h = max_bbox
            cv2.rectangle(vis_img, (x, y), (x+w, y+h), (0, 0, 255), 3)
            cv2.putText(vis_img, "Max Cluster", (x, max(y-10, 20)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        # Display prediction result
        text = f"Prediction: Wear Level {prediction}"
        cv2.rectangle(vis_img, (10, 10), (450, 60), (0, 0, 0), -1)
        cv2.putText(vis_img, text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        
        # Convert binary mask to BGR for concatenation
        mask_color = cv2.cvtColor(clean_mask, cv2.COLOR_GRAY2BGR)
        
        # Concatenate original image (with annotations) and mask
        combined_img = np.hstack((vis_img, mask_color))
        
        # Save visualization result
        out_file = vis_path / f"result_{file_path.name}"
        cv2.imencode('.png', combined_img)[1].tofile(str(out_file))
        
        print(f"Processed: {file_path.name} -> Predicted Level: {prediction}")
        
    print("-" * 40)
    print(f"Visualization complete. Results saved to: {vis_path}")

# ================= Execution Configuration =================
if __name__ == "__main__":
    # Define relative paths for the repository structure
    DATASET_CSV = "./labeled_dataset.csv"
    MODEL_PATH = "./rf_model.pkl"
    SCALER_PATH = "./scaler.pkl"
    
    # Directories for testing
    UNSEEN_IMAGES_DIR = "./test_images"
    OUTPUT_VIS_DIR = "./output_visualizations"
    
    # Ensure test directories exist
    Path(UNSEEN_IMAGES_DIR).mkdir(parents=True, exist_ok=True)
    
    # Execute training
    if Path(DATASET_CSV).exists():
        train_and_save_model(DATASET_CSV, MODEL_PATH, SCALER_PATH)
    else:
        print(f"Warning: {DATASET_CSV} not found. Skipping training.")
    
    # Execute testing and visualization
    if Path(UNSEEN_IMAGES_DIR).exists() and any(Path(UNSEEN_IMAGES_DIR).iterdir()):
        test_and_visualize_unseen(UNSEEN_IMAGES_DIR, OUTPUT_VIS_DIR, MODEL_PATH, SCALER_PATH)
    else:
        print(f"Warning: No images found in {UNSEEN_IMAGES_DIR}. Skipping testing.")
